# Generate synthetic data for the agnews dataset
## DeepSeek-R1-Distill-Qwen-1.5B

https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

1. baseline
2. targeted + linguistic tags
3. unsupervised context
4. (unsupervised context + linguistic tags)

as seen in the documentation we do:
- Avoid adding a system prompt; all instructions should be contained within the user prompt.
- To ensure that the model engages in thorough reasoning, we recommend enforcing the model to initiate its response with "<think>\n" at the beginning of every output.

Both of this are done via the `tokenizer.chat_template()` method

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# CHANGE WORKING DIRECTORY TO ROOT
current_dir = os.path.basename(os.getcwd())
if current_dir == "src":
    os.chdir("..")
elif os.path.basename(os.getcwd()) == "bai-thesis-nlp":  
    pass
else:
    os.chdir("../..")
from src._utils._generate_dataset import main_generate_dataset
from src._utils._helpers import get_generated_examples_df, clear_cuda_cache, get_context_examples

# get true labels
df_real = pd.read_csv("real_data/train/agnewstrainAll.csv").rename(
    columns={"2": "text", "3": "label"}
)
correct_labels = df_real["label"].unique().tolist()
labels_str = ", ".join(correct_labels)
labels_str_bullet = "\n".join([f"- {name}" for name in correct_labels])
labels_str_bullet_bold = "\n".join([f"- **{name}**" for name in correct_labels])
model = None
HF_TOKEN = open("src/_utils/hf_token.txt","r").read() # your huggingface token

In [ ]:
PROMPTS = {}

PROMPTS["baseline"] = f"""\
You are an expert in journalism and NLP specializing in news classification. \
Your task is to generate 10 high-quality short documents, that talks about the following four News categories:  
{labels_str_bullet_bold}

### **Output Format (JSON)**  
Return only a valid JSON list of 10 items in the following structure:

```json
[
    {{"text": <text>, "label": <label>}},
    ...
]
```
"""


PROMPTS["targeted + linguistic tags"] = f"""\
You are an expert in journalism and NLP specializing in news classification. \
Your task is to generate 10 high-quality short documents, that talks about the following four News categories (labels):  
{labels_str_bullet_bold}

For each example, also list the key phenomena it covers.

### **Follow these topics:**
- **Business**  
  - Markets  
  - Economy  
  - Companies  
  - Startups  
  - Regulations  

- **Sci/Tech**  
  - AI  
  - Space  
  - Cybersecurity  
  - Biotech  
  - Climate  

- **Sports**  
  - Events  
  - Records  
  - Highlights  
  - Scandals  
  - Olympics  

- **World**  
  - Politics  
  - Conflicts  
  - Disasters  
  - Human Rights  
  - Trade

### **Output Format (JSON)**
The labels must be one of the specified categories, which are: Business, Sci/Tech, Sports, World.
Return only a valid JSON list of 10 elements in the following structure:

```json
[
    {{"text": <text of the document>, "label": <corresponding label>, "phenomena": ["<phenomenon1>", "<phenomenon2>", ...]}},
    ...
]
```
"""


PROMPTS["unsupervised context"] = (
# baseline prompt
f"""\
You are an expert in journalism and NLP specialized in news classification. \
Your task is to generate an high-quality short documents, that talks about one of the following four News categories (labels):
{labels_str_bullet}

Here some examples of the documents you can use as a reference:
""",
# postfix
"""
Generate a new news document, with the corresponding category (label) with the following format:
```json
[
    {
        "text": "<text of the document>", 
        "label": "<corresponding label>",
    }
]
```
"""
)

# DeepSeek-R1-Distill-Qwen-1.5B

In [4]:
#############################################
# LOAD MODEL
#############################################

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
quantization_config = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="cuda",
    attn_implementation="flash_attention_2",
    quantization_config=quantization_config,
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
OUTPUT_DIR = "synthetic_data/datasets/DeepSeek-R1-Distill-Qwen-1.5B/"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)


base_config = {
    "dataset": "agnews",
    "model": model,
    "tokenizer": tokenizer,
    # "generation_method": "baseline",
    # "prompt": prompt,
    "system_prompt": None,
    "apply_chat_template": True,
    "num_examples": 500,
    "max_new_tokens": 4096,
    "seed": 42,
    #"json_output_file": "synthetic_data/datasets/DeepSeek-R1-Distill-Qwen-1.5B/agnews_baseline_500.json",
    "log_file": OUTPUT_DIR+"generate_dataset_agnews_log.json",
    "correct_labels": correct_labels,
    "correct_fields": ["text", "label"],
    ### context
    # "context_examples": None,
    # "prompt_postfix": None,
}

### 1. baseline

In [ ]:
name = "baseline"
config = base_config.copy()
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"agnews_baseline_500.json"
main_generate_dataset(config)

### 2. targeted + linguistic tags

In [ ]:
config = base_config.copy()
name = "targeted + linguistic tags"
config["generation_method"] = name
config["prompt"] = PROMPTS[name]
config["json_output_file"] = OUTPUT_DIR+"agnews_targeted+tags_500.json"
config["correct_fields"] = ["text", "label", "phenomena"]
main_generate_dataset(config)

In [ ]:
### Generate other 500 examples
config['seed'] = config['seed']*8
config['json_output_file'] = OUTPUT_DIR+"agnews_targeted+tags_500_2.json"

main_generate_dataset(config)


🚀 Starting Synthetic Dataset Generation
📊 Dataset              : agnews
📚 Generation method    : targeted + linguistic tags
🤖 Model                : deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
🔢 Examples to Generate : 500
💾 Output File          : synthetic_data/datasets/syn_agnews_targeted+tags_500_2.json
🕹️  Max New Tokens       : 4096
🎯 Seed                 : 336



Generating Examples:  28%|██▊       | 141/500 [04:25<10:19,  1.73s/ex, examples=141/500, run=11]

❌ Failed to parse generation 11: Expecting value: line 1 column 1 (char 0)


Generating Examples:  28%|██▊       | 141/500 [04:48<10:19,  1.73s/ex, examples=141/500, run=12]

❌ Failed to parse generation 12: Expecting ',' delimiter: line 4 column 9 (char 271)


...

❌ Failed to parse generation 43: Expecting value: line 1 column 1 (char 0)


Generating Examples:  93%|█████████▎| 465/500 [21:43<01:29,  2.56s/ex, examples=465/500, run=48]

❌ Failed to parse generation 48: Expecting value: line 1 column 1 (char 0)


Generating Examples: 100%|██████████| 500/500 [23:00<00:00,  2.76s/ex, examples=500/500, run=52]

📝 Log saved successfully to: src/agnews/generate_dataset_agnews_log.json
💾 Dataset with metadata saved to: synthetic_data/datasets/syn_agnews_targeted+tags_500_2.json


### 3. unsupervised context

In each prompt we attach n (5) examples sampled randomly from the train set. The samples are used without the labels, so they works as unsupervised context for the model, when we will generate the new synthetic sample. 

In [7]:
# we take more than 500 because it can happen that some prompt
# generate the example in the wrong format, so is not read correctly (and discarded)
num_prompts = 1000
num_examples_per_prompt = 5
np.random.seed(42)
context_examples = get_context_examples(df_real, num_examples_per_prompt, num_prompts)
print(f"Number prompts: {len(context_examples)}")
print(f"Number of examples per prompt: {len(context_examples[0])}")
print(context_examples[0])

Number prompts: 1000
Number of examples per prompt: 5
['With Derek Lowe #39;s days in Boston almost certainly numbered -- and Pedro Martinez #39;s future here in question -- the Red Sox are expected to greet All-Star righthander Carl Pavano this week on Yawkey Way.', 'Symantec has released firmware fixes for a string of critical security holes in its firewall/VPN and Gateway Security products which could be exploited to cause a denial of service, identify ', 'King County prosecutors charged a Covington orthodontist yesterday with engaging in sexually explicit Internet conversations with several girls, including three current or former patients, and with dealing child pornography.', 'LONDON (Reuters) - Oil producers are emerging to lock in record high prices for their future crude output, but activity is modest as firms still fear calling a premature end to this year #39;s stunning price rise, traders said on Friday. ', 'STOCKHOLM (AFP) - Andre Agassi was set to intensify his chase for 

In [ ]:
config = base_config.copy()
name = "unsupervised context"
config["generation_method"] = name
config["prompt"] = PROMPTS[name][0]
config["prompt_postfix"] = PROMPTS[name][1]
config["max_new_tokens"] = 2048
config["json_output_file"] = OUTPUT_DIR+"agnews_unsupervisedContext_500.json"
config["context_examples"] = context_examples
main_generate_dataset(config)

In [ ]:
generated_df, _ = get_generated_examples_df(OUTPUT_DIR+"agnews_unsupervisedContext_500.json")

for i in range(5):
    print("TEXT: "+generated_df.iloc[i]['text'])
    print("LABEL: "+generated_df.iloc[i]['label'])
    print("CONTEXT EXAMPLES:")
    for j in range(len(generated_df.iloc[i]['context_examples'])):
        print("- "+generated_df.iloc[i]['context_examples'][j])

    print("\n"+"=="*50)

TEXT: A tech company is launching a new product line to boost revenue and set the stage for future growth. The launch is expected to generate significant profits and position the company as a leader in the industry.
LABEL: Business
CONTEXT EXAMPLES:
- With Derek Lowe #39;s days in Boston almost certainly numbered -- and Pedro Martinez #39;s future here in question -- the Red Sox are expected to greet All-Star righthander Carl Pavano this week on Yawkey Way.
- Symantec has released firmware fixes for a string of critical security holes in its firewall/VPN and Gateway Security products which could be exploited to cause a denial of service, identify 
- King County prosecutors charged a Covington orthodontist yesterday with engaging in sexually explicit Internet conversations with several girls, including three current or former patients, and with dealing child pornography.
- LONDON (Reuters) - Oil producers are emerging to lock in record high prices for their future crude output, but acti

In [ ]:
# #############################################
# # GENERATE UNSUPERVISED CONTEXT + TAGS AGNEWS DATASET
# #############################################

# base_prompt = """\
# You are an expert in journalism and NLP specialized in news classification. \
# Your task is to generate an high-quality short documents, that talks about one of the following four News categories (labels):
# - Business
# - Sci/Tech
# - Sports
# - World.

# Here some examples of the documents you can use as a reference:
# """
# postfix = """
# Generate a new news document, with the corresponding category (label) and the key linguistic phenomena it covers, with the following format:
# ```json
# [
#     {
#         "text": "<text of the document>", 
#         "label": "<corresponding label>", 
#         "phenomena": ["<phenomenon1>", "<phenomenon2>", ...] 
#     }
# ]
# ```
# """
# config = base_config.copy()
# config["generation_method"] = "unsupervised context + linguistic tags"
# config["prompt"] = base_prompt
# config["max_new_tokens"] = 2048
# config["json_output_file"] = OUTPUT_DIR+"agnews_unsupervisedContext+tags_500.json"
# config["context_examples"] = context_examples
# config["prompt_postfix"] = postfix
# config["correct_fields"] = ["text", "label", "phenomena"]
# config["num_examples"]= 5
# main_generate_dataset(config)

In [ ]:
# generated_df, _ = get_generated_examples_df(OUTPUT_DIR+"agnews_unsupervisedContext+tags_500.json")

# for i in range(5):
#     print("TEXT: "+generated_df.iloc[i]['text'])
#     print("LABEL: "+generated_df.iloc[i]['label'])
#     print("CONTEXT EXAMPLES:")
#     for j in range(len(generated_df.iloc[i]['context_examples'])):
#         print("- "+generated_df.iloc[i]['context_examples'][j])
#     print("PHENOMENA:", generated_df.iloc[i]['phenomena'])

#     print("\n"+"=="*50)